# Data Preparation — Deteção de Conexões Maliciosas (IAAC — Grupo 10)

Split estratificado, feature engineering, pipeline de pré-processamento (`ColumnTransformer`), tratamento do desbalanceamento e um baseline (Random Forest) para validar o pipeline de ponta a ponta.


In [1]:
import pandas as pd, numpy as npfrom sklearn.model_selection import train_test_splitfrom sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformerfrom sklearn.compose import ColumnTransformerfrom sklearn.pipeline import Pipelinefrom sklearn.impute import SimpleImputerfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.metrics import recall_score, precision_score, roc_auc_score, average_precision_score, confusion_matrixfrom sklearn.utils.class_weight import compute_class_weightdf = pd.read_csv("../datasets/Raw/cybersecurity_network_logs.csv")target = "Is_Malicious"num_cols_log = ["Packet_Size_Bytes", "Geo_Distance_km"]       # muito assimétricas -> log1pnum_cols_plain = ["Connection_Duration_ms", "Failed_Logins"]  # mantidas na escala originalcat_cols = ["Protocol"]X = df.drop(columns=[target])y = df[target]

## 1. Split estratificado (antes de qualquer transformação)

70% treino / 15% validação / 15% teste, com `stratify=y` para preservar os 4,99% de maliciosos em cada partição, e semente fixa para reprodutibilidade.


In [1]:
X_train, X_temp, y_train, y_temp = train_test_split(    X, y, test_size=0.30, stratify=y, random_state=42)X_val, X_test, y_val, y_test = train_test_split(    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)for name, (Xs, ys) in {"train": (X_train,y_train), "val": (X_val,y_val), "test": (X_test,y_test)}.items():    print(f"{name}: n={len(Xs)}, maliciosos={int(ys.sum())}, taxa={ys.mean():.2%}")

train: n=17500, maliciosos=873, taxa=4.99%
val: n=3750, maliciosos=187, taxa=4.99%
test: n=3750, maliciosos=187, taxa=4.99%


Prevalência mantida em todas as partições (~4,99%). No teste ficam 187 positivos — bem acima do mínimo de ~30 recomendado para uma avaliação estável.


## 2. Feature engineering

A EDA mostrou que `Failed_Logins >= 3` já é quase perfeitamente discriminativo (93–100% maliciosas). Criamos essa feature binária como sinal explícito para o modelo, mantendo também o valor contínuo original.


In [1]:
def add_features(df_in):    df2 = df_in.copy()    df2["High_Failed_Logins"] = (df2["Failed_Logins"] >= 3).astype(int)    return df2X_train_fe = add_features(X_train)X_val_fe   = add_features(X_val)X_test_fe  = add_features(X_test)num_cols_plain_fe = num_cols_plain + ["High_Failed_Logins"]X_train_fe[["Failed_Logins","High_Failed_Logins"]].head()

## 3. Pipeline de pré-processamento (`ColumnTransformer`)

- **`Packet_Size_Bytes`, `Geo_Distance_km`** (skew ≈4): imputação por mediana → `log1p` → `StandardScaler`. `log1p(0) = 0`, por isso os zeros (17 e 12 casos no treino) são tratados sem problema.
- **`Connection_Duration_ms`, `Failed_Logins`, `High_Failed_Logins`**: imputação por mediana → `StandardScaler` (sem log, distribuição menos extrema / já binária).
- **`Protocol`**: imputação pela moda → `OneHotEncoder` (mantido apesar do sinal fraco isolado, para permitir interações no modelo).

O `preprocessor` é ajustado (`fit`) **só no treino**, para evitar fuga de informação (data leakage) do val/teste.


In [1]:
log_pipeline = Pipeline(steps=[    ("impute", SimpleImputer(strategy="median")),    ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),    ("scale", StandardScaler()),])plain_pipeline = Pipeline(steps=[    ("impute", SimpleImputer(strategy="median")),    ("scale", StandardScaler()),])cat_pipeline = Pipeline(steps=[    ("impute", SimpleImputer(strategy="most_frequent")),    ("onehot", OneHotEncoder(handle_unknown="ignore")),])preprocessor = ColumnTransformer(transformers=[    ("log_num", log_pipeline, num_cols_log),    ("plain_num", plain_pipeline, num_cols_plain_fe),    ("cat", cat_pipeline, cat_cols),])preprocessor.fit(X_train_fe)   # fit só no treinoX_train_t = preprocessor.transform(X_train_fe)X_val_t   = preprocessor.transform(X_val_fe)X_test_t  = preprocessor.transform(X_test_fe)print("Features após pré-processamento:", preprocessor.get_feature_names_out())print("Shape treino:", X_train_t.shape, "| val:", X_val_t.shape, "| teste:", X_test_t.shape)

Features após pré-processamento: ['log_num__Packet_Size_Bytes' 'log_num__Geo_Distance_km'
 'plain_num__Connection_Duration_ms' 'plain_num__Failed_Logins'
 'plain_num__High_Failed_Logins' 'cat__Protocol_ICMP' 'cat__Protocol_TCP'
 'cat__Protocol_UDP']
Shape treino: (17500, 8) | val: (3750, 8) | teste: (3750, 8)


## 4. Desbalanceamento (4,99% de positivos)

Em vez de reamostragem (oversampling/undersampling), usamos `class_weight="balanced"` — mais simples, sem risco de duplicar/perder informação, e ajustável depois por threshold. Os pesos calculados:


In [1]:
classes = np.unique(y_train)weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)print(dict(zip(classes, weights)))print("Contagem no treino:", y_train.value_counts().to_dict())

{0: 0.5262524809045528, 1: 10.02290950744559}
Contagem no treino: {0: 16627, 1: 873}


Um caso malicioso "pesa" ~10x mais que um caso benigno no ajuste do modelo — compensa a raridade sem alterar os dados. **Avaliação sempre por recall / precisão / PR-AUC, nunca por accuracy** (accuracy fica ~95% mesmo prevendo sempre "benigno").


## 5. Baseline: Pipeline completo (pré-processamento + Random Forest)

Só para validar o pipeline de ponta a ponta — a escolha e afinação do modelo final é o próximo passo do roadmap, não este.


In [1]:
full_pipeline = Pipeline(steps=[    ("preprocess", preprocessor),    ("model", RandomForestClassifier(        n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1    )),])full_pipeline.fit(X_train_fe, y_train)y_val_pred = full_pipeline.predict(X_val_fe)y_val_proba = full_pipeline.predict_proba(X_val_fe)[:,1]print("Recall:   ", round(recall_score(y_val, y_val_pred), 4))print("Precisão: ", round(precision_score(y_val, y_val_pred), 4))print("ROC-AUC:  ", round(roc_auc_score(y_val, y_val_proba), 4))print("PR-AUC:   ", round(average_precision_score(y_val, y_val_proba), 4))cm = confusion_matrix(y_val, y_val_pred)tn, fp, fn, tp = cm.ravel()print("\nMatriz de confusão (val):\n", cm)print(f"Taxa de falsos positivos: {fp/(fp+tn):.4%}")

Recall:    0.893
Precisão:  0.9766
ROC-AUC:   0.9599
PR-AUC:    0.917

Matriz de confusão (val):
 [[3559    4]
 [  20  167]]
Taxa de falsos positivos: 0.1123%


## 6. Leitura face aos critérios de sucesso definidos no Business Understanding

| Critério | Alvo | Baseline (validação) | Estado |
|---|---|---|---|
| Recall | ≥ 90% | 89,3% | Muito perto — falta afinar threshold/modelo |
| Taxa de falsos positivos | ≤ 1% | 0,11% | Cumprido com folga |
| PR-AUC | — | 0,917 | Bom para um baseline sem tuning |

O recall está ligeiramente abaixo do alvo (89,3% vs 90%) com o threshold por omissão (0,5). Como a taxa de FP está muito abaixo do limite (0,11% vs 1%), há margem para **baixar o threshold de decisão** e recuperar recall sem violar o critério de falsos positivos — é o primeiro ajuste a fazer na fase de modelação.

**Objeto reutilizável:** `preprocessor` (e `full_pipeline`) ficam prontos para serem usados/exportados (`joblib.dump`) e reaproveitados nas próximas fases (tuning de modelo, avaliação final no teste).
